In [ ]:
from snowflake.snowpark.context import get_active_session
from datetime import datetime
from snowflake.snowpark.functions import col, trim, lower, lit, upper, regexp_replace, coalesce, when, row_number, sha2, concat_ws, abs
from snowflake.snowpark.window import Window
session = get_active_session()

with open('../DWH/parameters.sql', 'r') as f:
    sql_content = f.read()

params = {}
for line in sql_content.split('\n'):
    line = line.strip()
    if line.upper().startswith('SET ') and '=' in line:
        line = line.rstrip(';').strip()
        key_val = line[4:].strip()          
        key, val = key_val.split('=', 1)
        key = key.strip()
        val = val.strip().strip("'")
        params[key] = val

# schemas
DB    = params.get('db')
WH    = params.get('wh')
RAW   = params.get('raw_schema')
STG   = params.get('stg_schema')
DWH   = params.get('dwh_schema')
AUDIT = params.get('audit_schema')
UTIL  = params.get('util_schema')

print(f"DB: {DB}")
print(f"WH: {WH}")

# streams
STREAM_PRODUCTS               = params.get('stream_products')
STREAM_CUSTOMERS              = params.get('stream_customers')
STREAM_CUSTOMER_ADDRESSES     = params.get('stream_customer_addresses')
STREAM_SUPPLIERS              = params.get('stream_suppliers')
STREAM_PRODUCT_SUPPLIERS      = params.get('stream_product_suppliers')
STREAM_STORES                 = params.get('stream_stores')
STREAM_EMPLOYEES              = params.get('stream_employees')
STREAM_ORDERS                 = params.get('stream_orders')
STREAM_ORDER_ITEMS            = params.get('stream_order_items')
STREAM_INVENTORY_TRANSACTIONS = params.get('stream_inventory_transactions')

# staging tables
STG_PRODUCTS               = params.get('stg_products')
STG_CUSTOMERS              = params.get('stg_customers')
STG_CUSTOMER_ADDRESSES     = params.get('stg_customer_addresses')
STG_SUPPLIERS              = params.get('stg_suppliers')
STG_PRODUCT_SUPPLIERS      = params.get('stg_product_suppliers')
STG_STORES                 = params.get('stg_stores')
STG_EMPLOYEES              = params.get('stg_employees')
STG_ORDERS                 = params.get('stg_orders')
STG_ORDER_ITEMS            = params.get('stg_order_items')
STG_INVENTORY_TRANSACTIONS = params.get('stg_inventory_transactions')
STG_INVALID_RECORDS        = params.get('stg_invalid_records')

# dwh tables
DIM_PRODUCT                   = params.get('dim_product')
DIM_CUSTOMER                  = params.get('dim_customer')
DIM_CUSTOMER_ADDRESS          = params.get('dim_customer_address')
DIM_SUPPLIER                  = params.get('dim_supplier')
BRIDGE_PRODUCT_SUPPLIER       = params.get('bridge_product_supplier')
DIM_STORE                     = params.get('dim_store')
DIM_EMPLOYEE                  = params.get('dim_employee')
FACT_ORDERS                   = params.get('fact_orders')
FACT_ORDER_ITEMS              = params.get('fact_order_items')
FACT_SALES                    = params.get('fact_sales')
FACT_INVENTORY_TRANSACTION    = params.get('fact_inventory_transaction')
FACT_DAILY_INVENTORY_SNAPSHOT = params.get('fact_daily_inventory_snapshot')

# stored procedures
SP_AUDIT  = params.get('sp_audit')
SP_NOTIFY = params.get('sp_notify')


In [ ]:
%%sql -r dataframe_2
USE DATABASE {{DB}};
USE WAREHOUSE {{WH}};

In [ ]:
print(f"DB: {DB}")
print(f"WH: {WH}")


In [ ]:
def has_data(stream):
     # return session.sql(f"SELECT SYSTEM$STREAM_HAS_DATA('RETAIL_ETL_DB_NEW.RAWDATA.{stream}')").collect()[0][0]
       return  session.call("SYSTEM$STREAM_HAS_DATA",f"{DB}.{RAW}.{stream}")

In [ ]:
if has_data("products_stream"):
    %run ../Staging/products.ipynb
    %run ../DWH/dwhproduct.ipynb
else:
    print("no streams data")

In [ ]:
if has_data("customers_stream"):
    %run ../Staging/customers.ipynb
    %run ../DWH/dwhcustomer.ipynb


In [ ]:
if has_data("customer_addresses_stream"):
    %run ../Staging/customeraddress.ipynb
    %run ../DWH/dwhcustomeraddresses.ipynb          

In [ ]:
if has_data("suppliers_stream"):
    %run ../Staging/suppliers.ipynb
    %run ../DWH/dwhsupplier.ipynb
   


In [ ]:
if has_data("product_suppliers_stream"):
    %run ../Staging/product_suppliers.ipynb
    %run ../DWH/bridgeproductsupplier.ipynb

In [ ]:
# if has_data("stores_stream"):
#     %run ../Staging/stores.ipynb
#     %run ../DWH/dwhstore.ipynb

if has_data("employees_stream"):
    %run ../Staging/employees.ipynb
    %run ../DWH/dwhemployee.ipynb


In [ ]:
if has_data("orders_stream"):
    %run ../Staging/orders.ipynb
    %run ../DWH/fact_orders.ipynb

In [ ]:
if has_data("order_items_stream"):
    %run ../Staging/order_items.ipynb
    %run ../DWH/fact_order_items.ipynb
    
    %run ../DWH/factsales.ipynb

In [ ]:
if has_data("inventory_transactions_stream"):
    %run ../Staging/inventory_transactions.ipynb
    %run ../DWH/factinventorytransaction.ipynb
    %run ../DWH/factdailyinventorysnapshot.ipynb
